## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import warnings
from datetime import datetime

from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# PyCaret imports
from pycaret.regression import (
    setup, compare_models, pull, save_model, load_model,
    predict_model, finalize_model, create_model
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuration
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_EXPERTS = 3  # Top 3 models per soil property
N_FOLDS_GATING = 5  # Folds for generating OOF predictions for gating network
RESULTS_DIR = './revision-29-04-26/results'
MODELS_DIR = './revision-29-04-26/saved_models'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Experiment started at: {datetime.now()}")
print(f"Random state: {RANDOM_STATE}")
print(f"Test size: {TEST_SIZE}")
print(f"Number of experts: {N_EXPERTS}")

## 2. Data Loading & Exploration

In [ ]:
# Load dataset
data_path = '../data/merged-data-all-continent.xlsx'
df = pd.read_excel(data_path)

print(f"Dataset shape: {df.shape}")

# Clean column names
df.columns = [str(x).strip() for x in df.columns]

# Define target parameters (17 soil properties)
target_params = [
    "pH_1-1_Water", "CEC(meq/100g)", "EC  μS/cm", "T.Nitrogen %",
    "T.Carbon %", "LOI %", "Clay %", "Sand %", "Silt %",
    "Calcium, ppm", "Copper, ppm", "Magnesium, ppm",
    "Phosphorus, ppm", "Potassium, ppm", "Sodium, ppm",
    "Sulfur, ppm", "Zinc, ppm"
]

# Define feature columns
pxrf_columns = ['K', 'Ca', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Ni', 'Cu', 'Zn', 'As', 'Rb', 'Sr', 'Zr', 'Ba', 'Pb']
spectral_columns = [col for col in df.columns if col.isnumeric()]

print(f"\nNumber of target parameters: {len(target_params)}")
print(f"Number of spectral columns: {len(spectral_columns)}")
print(f"Number of pXRF columns: {len(pxrf_columns)}")

# Print sample counts per target
print("\n--- Sample counts per target parameter ---")
for param in target_params:
    count = df[param].notnull().sum()
    print(f"  {param}: {count} rows with data")

## 3. Data Preparation

**Critical change**: We now:
1. Prepare target-specific DataFrames (filtering NaN rows)
2. Split into train/test FIRST
3. Pass ONLY training data to PyCaret's `setup()`
4. PyCaret applies transformations only on train, transforms test correctly

In [ ]:
def prepare_data_for_target(df, target_param, pxrf_columns, spectral_columns):
    """
    Prepare data for a specific target parameter.
    Returns the clean dataframe (NaN-free) WITHOUT any transformations.
    Transformations will be applied later ONLY on training data via PyCaret.
    
    Also removes constant/near-constant feature columns that cause
    BracketError in Yeo-Johnson transformation (scipy brent optimizer
    fails when a feature has zero variance).
    """
    # Get rows where target is not null
    target_df = df[df[target_param].notnull()].copy()
    
    # Select features and target
    features = spectral_columns + pxrf_columns
    target_df = target_df[['Sample ID', target_param] + features]
    
    # Convert all feature columns to numeric (coerce errors to NaN)
    for col in features:
        target_df[col] = pd.to_numeric(target_df[col], errors='coerce')
    
    # Remove rows with any missing values
    target_df = target_df.dropna()
    
    # Remove constant or near-constant feature columns
    # These cause BracketError in Yeo-Johnson power transformation
    low_var_threshold = 1e-10
    cols_to_drop = []
    for col in features:
        if col in target_df.columns:
            col_var = target_df[col].var()
            if col_var < low_var_threshold:
                cols_to_drop.append(col)
    
    if cols_to_drop:
        print(f"  Removing {len(cols_to_drop)} constant/near-constant features")
        target_df = target_df.drop(columns=cols_to_drop)
    
    remaining_features = [c for c in features if c in target_df.columns]
    
    print(f"Data prepared for {target_param}:")
    print(f"  - Samples: {len(target_df)}")
    print(f"  - Features: {len(remaining_features)}")
    print(f"  - Target range: [{target_df[target_param].min():.2f}, {target_df[target_param].max():.2f}]")
    print(f"  - Target mean: {target_df[target_param].mean():.2f} ± {target_df[target_param].std():.2f}")
    
    return target_df

## 4.Individual Expert Training Pipeline

PyCaret `setup()` is called ONLY on `train_df`.
The test set is held out completely during training and preprocessing.

In [ ]:
def train_individual_experts(df, target_param, pxrf_columns, spectral_columns,
                              n_experts=3, random_state=42, test_size=0.2,
                              models_dir='./revision-29-04-26/saved_models'):
    print(f"\n{'='*70}")
    print(f"Training experts for: {target_param}")
    print(f"{'='*70}")
    
    # Step 1: Prepare clean data (also removes constant features)
    target_df = prepare_data_for_target(df, target_param, pxrf_columns, spectral_columns)
    
    if len(target_df) < 50:
        print(f"  WARNING: Insufficient data ({len(target_df)} samples). Skipping.")
        return None
    
    # Step 2: CRITICAL - Split FIRST, before any preprocessing
    train_df, test_df = train_test_split(
        target_df, test_size=test_size, random_state=random_state
    )
    print(f"\n  Train set: {len(train_df)} samples")
    print(f"  Test set:  {len(test_df)} samples")
    
    # Step 2b: After split, remove any features that become constant in train set
    # (the split can create new constant columns that weren't constant in the full data)
    feature_cols = [c for c in train_df.columns if c not in ['Sample ID', target_param]]
    low_var_cols = [c for c in feature_cols if train_df[c].var() < 1e-10]
    if low_var_cols:
        print(f"  Removing {len(low_var_cols)} features that became constant after split")
        train_df = train_df.drop(columns=low_var_cols)
        test_df = test_df.drop(columns=low_var_cols)
    
    # Step 3: PyCaret setup() on TRAINING DATA ONLY
    # This ensures all transformations (normalize, Yeo-Johnson, feature selection)
    # are fit on training data only. Test data is never seen during fitting.
    # If Yeo-Johnson fails (BracketError), fall back to transformation=False.
    use_transformation = True
    try:
        s = setup(
            data=train_df,
            target=target_param,
            ignore_features=['Sample ID'],
            session_id=random_state,
            normalize=True,
            transformation=True,
            feature_selection=True,
            system_log=False,
            verbose=False
        )
    except Exception as e:
        print(f"  WARNING: PyCaret setup with transformation=True failed: {e}")
        print(f"  Retrying with transformation=False...")
        use_transformation = False
        s = setup(
            data=train_df,
            target=target_param,
            ignore_features=['Sample ID'],
            session_id=random_state,
            normalize=True,
            transformation=False,
            feature_selection=True,
            system_log=False,
            verbose=False
        )
    
    print(f"  Transformation enabled: {use_transformation}")
    
    # Step 4: Compare models, select top N
    best_models = compare_models(n_select=n_experts)
    cv_results = pull()
    
    # Step 5: Save models
    safe_param = target_param.lower().replace('/', '_').replace('%', '').replace(' ', '_').replace('(', '').replace(')', '').replace(',', '')
    model_dir = os.path.join(models_dir, safe_param)
    os.makedirs(model_dir, exist_ok=True)
    
    if not isinstance(best_models, list):
        best_models = [best_models]
    
    for i, model in enumerate(best_models):
        save_model(model, os.path.join(model_dir, f'model_{i+1}'))
    
    # Step 6: Evaluate on held-out test set
    test_results = {}
    for i, model in enumerate(best_models):
        predictions = predict_model(model, data=test_df)
        pred_col = 'prediction_label'
        
        y_true = predictions[target_param].values
        y_pred = predictions[pred_col].values
        
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        bias = np.mean(y_pred - y_true)
        
        test_results[f'expert_{i+1}'] = {
            'R2': r2, 'RMSE': rmse, 'MAE': mae, 'Bias': bias,
            'model_type': type(model).__name__
        }
        print(f"  Expert {i+1} ({type(model).__name__}): R²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, Bias={bias:.4f}")
    
    return {
        'target_param': target_param,
        'train_df': train_df,
        'test_df': test_df,
        'models': best_models,
        'cv_results': cv_results,
        'test_results': test_results,
        'model_dir': model_dir,
        'transformation_used': use_transformation
    }

## 5. Mixture of Experts Architecture

Gating network is trained on OUT-OF-FOLD predictions,
NOT on in-sample predictions from expert models.

In [ ]:
class CorrectedMixtureOfExperts:
    
    def __init__(self, target_parameter, n_folds=5, random_state=42):
        self.target_parameter = target_parameter
        self.experts = []
        self.gating_network = None
        self.scaler = StandardScaler()
        self.is_fitted = False
        self.n_folds = n_folds
        self.random_state = random_state
        self.expert_weights = None
        
    def set_experts(self, expert_models):
        """Set the pre-trained expert models."""
        self.experts = expert_models
        print(f"Set {len(self.experts)} experts for {self.target_parameter}")
    
    def train_gating_network_oof(self, X_train, y_train):
        if len(self.experts) == 0:
            raise ValueError("No experts set. Call set_experts() first.")
        
        n_samples = len(X_train)
        n_experts = len(self.experts)
        
        # Generate OOF predictions for each expert
        oof_predictions = np.zeros((n_samples, n_experts))
        
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        
        for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train)):
            print(f"  Gating OOF fold {fold_idx + 1}/{self.n_folds}...")
            
            X_fold_train = X_train.iloc[train_idx]
            y_fold_train = y_train.iloc[train_idx]
            X_fold_val = X_train.iloc[val_idx]
            
            for expert_idx, expert in enumerate(self.experts):
                try:
                    # Clone the expert and retrain on this fold's training data
                    from sklearn.base import clone
                    fold_expert = clone(expert)
                    fold_expert.fit(X_fold_train, y_fold_train)
                    
                    # Predict on validation fold (OOF)
                    pred = fold_expert.predict(X_fold_val)
                    oof_predictions[val_idx, expert_idx] = pred
                except Exception as e:
                    print(f"    Error with expert {expert_idx} on fold {fold_idx}: {e}")
                    oof_predictions[val_idx, expert_idx] = y_train.iloc[val_idx].mean()
        
        # Calculate expert performance weights from OOF predictions
        expert_weights = []
        for i in range(n_experts):
            mse = mean_squared_error(y_train, oof_predictions[:, i])
            weight = 1.0 / (mse + 1e-8)
            expert_weights.append(weight)
        
        expert_weights = np.array(expert_weights)
        expert_weights = expert_weights / np.sum(expert_weights)
        self.expert_weights = expert_weights
        
        # Train gating network on OOF predictions
        # The gating network learns to select the best expert for each sample
        # based on the input features.
        X_scaled = self.scaler.fit_transform(X_train)
        
        # Create per-sample targets: which expert is best?
        gating_targets = []
        for i in range(n_samples):
            errors = [abs(oof_predictions[i, j] - y_train.iloc[i]) for j in range(n_experts)]
            best_expert = np.argmin(errors)
            target = np.zeros(n_experts)
            target[best_expert] = 1.0
            gating_targets.append(target)
        
        gating_targets = np.array(gating_targets)
        
        self.gating_network = MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            max_iter=500,
            random_state=self.random_state
        )
        
        self.gating_network.fit(X_scaled, gating_targets)
        self.is_fitted = True
        
        print(f"  Expert weights (from OOF): {expert_weights}")
        return expert_weights
    
    def predict(self, X):
        """Make predictions using the mixture of experts."""
        if not self.is_fitted:
            raise ValueError("Model not fitted. Call train_gating_network_oof() first.")
        
        # Get expert predictions
        expert_predictions = []
        for expert in self.experts:
            try:
                pred = expert.predict(X)
                expert_predictions.append(pred)
            except Exception as e:
                print(f"Error in expert prediction: {e}")
                expert_predictions.append(np.zeros(len(X)))
        
        expert_predictions = np.column_stack(expert_predictions)
        
        # Get gating weights
        X_scaled = self.scaler.transform(X)
        gating_weights = self.gating_network.predict(X_scaled)
        
        # Ensure weights are non-negative and sum to 1
        gating_weights = np.maximum(gating_weights, 0)
        row_sums = np.sum(gating_weights, axis=1, keepdims=True)
        gating_weights = gating_weights / (row_sums + 1e-8)
        
        # Combine predictions
        final_predictions = np.sum(expert_predictions * gating_weights, axis=1)
        
        return final_predictions, gating_weights
    
    def evaluate(self, X, y):
        """Evaluate the mixture of experts model."""
        predictions, weights = self.predict(X)
        
        mse = mean_squared_error(y, predictions)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y, predictions)
        r2 = r2_score(y, predictions)
        bias = np.mean(predictions - y)
        
        return {
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2,
            'Bias': bias,
            'predictions': predictions,
            'gating_weights': weights
        }

## 6. Run Full Pipeline for All Soil Properties

In [ ]:
# Store all results
all_individual_results = {}
all_moe_results = {}

for param in target_params:
    # ---- Step A: Train individual experts ----
    result = train_individual_experts(
        df, param, pxrf_columns, spectral_columns,
        n_experts=N_EXPERTS, random_state=RANDOM_STATE,
        test_size=TEST_SIZE, models_dir=MODELS_DIR
    )
    
    if result is None:
        print(f"Skipped {param} (insufficient data)")
        continue
    
    all_individual_results[param] = result
    
    # ---- Step B: Train MoE with corrected gating ----
    print(f"\n  Training Corrected MoE for {param}...")
    
    train_df = result['train_df']
    test_df = result['test_df']
    features = spectral_columns + pxrf_columns
    # Filter to features that exist in the dataframe
    available_features = [f for f in features if f in train_df.columns]
    
    X_train = train_df[available_features]
    y_train = train_df[param]
    X_test = test_df[available_features]
    y_test = test_df[param]
    
    # Get the underlying sklearn models from PyCaret pipelines
    expert_models = result['models']
    
    # Create and train MoE
    moe = CorrectedMixtureOfExperts(
        target_parameter=param,
        n_folds=N_FOLDS_GATING,
        random_state=RANDOM_STATE
    )
    
    # For the gating network, we need the raw sklearn estimators
    # PyCaret models include a pipeline; we'll use predict_model for test predictions
    # But for OOF training of the gating, we need sklearn-compatible models
    # So we train simple sklearn models as experts for the gating
    from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor
    
    # Train simple sklearn expert models on training data for gating
    sklearn_experts = []
    for i in range(min(N_EXPERTS, 3)):
        if i == 0:
            expert = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
        elif i == 1:
            expert = GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
        else:
            expert = ExtraTreesRegressor(n_estimators=100, random_state=RANDOM_STATE)
        
        expert.fit(X_train, y_train)
        sklearn_experts.append(expert)
    
    moe.set_experts(sklearn_experts)
    expert_weights = moe.train_gating_network_oof(X_train, y_train)
    
    # Evaluate MoE on test set
    moe_eval = moe.evaluate(X_test, y_test)
    
    print(f"\n  MoE Results for {param}:")
    print(f"    R²:   {moe_eval['R2']:.4f}")
    print(f"    RMSE: {moe_eval['RMSE']:.4f}")
    print(f"    MAE:  {moe_eval['MAE']:.4f}")
    print(f"    Bias: {moe_eval['Bias']:.4f}")
    
    all_moe_results[param] = {
        'moe_model': moe,
        'evaluation': moe_eval,
        'expert_weights': expert_weights
    }

## 7. Results Summary & Export

In [ ]:
# Create comprehensive results table
results_rows = []

for param in target_params:
    if param not in all_individual_results:
        continue
    
    ind_result = all_individual_results[param]
    moe_result = all_moe_results.get(param)
    
    # Best individual expert test results
    best_expert = None
    best_r2 = -np.inf
    for expert_name, expert_res in ind_result['test_results'].items():
        if expert_res['R2'] > best_r2:
            best_r2 = expert_res['R2']
            best_expert = expert_name
    
    best_exp_res = ind_result['test_results'][best_expert]
    
    row = {
        'Parameter': param,
        'N_samples': len(ind_result['train_df']) + len(ind_result['test_df']),
        'N_train': len(ind_result['train_df']),
        'N_test': len(ind_result['test_df']),
        'Best_Expert': best_expert,
        'Best_Expert_Model': best_exp_res.get('model_type', 'N/A'),
        'Expert_R2': best_exp_res['R2'],
        'Expert_RMSE': best_exp_res['RMSE'],
        'Expert_MAE': best_exp_res['MAE'],
        'Expert_Bias': best_exp_res['Bias'],
    }
    
    if moe_result:
        moe_eval = moe_result['evaluation']
        row.update({
            'MoE_R2': moe_eval['R2'],
            'MoE_RMSE': moe_eval['RMSE'],
            'MoE_MAE': moe_eval['MAE'],
            'MoE_Bias': moe_eval['Bias'],
            'MoE_Improvement_R2': moe_eval['R2'] - best_exp_res['R2']
        })
    
    results_rows.append(row)

results_df = pd.DataFrame(results_rows)
print("\n" + "="*80)
print("COMPREHENSIVE RESULTS SUMMARY")
print("="*80)
print(results_df.to_string(index=False))

# Save to Excel
results_path = os.path.join(RESULTS_DIR, 'revised_individual_moe_results.xlsx')
results_df.to_excel(results_path, index=False)
print(f"\nResults saved to: {results_path}")

## 8. Visualization: R² vs. RMSE Chart (Country-Color-Coded)

In [ ]:
# R² comparison chart
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Plot 1: Best Expert R² vs MoE R² for each parameter
if len(results_df) > 0 and 'MoE_R2' in results_df.columns:
    params = results_df['Parameter'].values
    expert_r2 = results_df['Expert_R2'].values
    moe_r2 = results_df['MoE_R2'].values
    
    x = np.arange(len(params))
    width = 0.35
    
    axes[0].bar(x - width/2, expert_r2, width, label='Best Expert', alpha=0.8, color='steelblue')
    axes[0].bar(x + width/2, moe_r2, width, label='MoE (Corrected)', alpha=0.8, color='coral')
    axes[0].set_xlabel('Soil Property')
    axes[0].set_ylabel('R² (Test Set)')
    axes[0].set_title('Individual Expert vs. Corrected MoE Performance')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(params, rotation=90, fontsize=8)
    axes[0].legend()
    axes[0].axhline(y=0.8, color='green', linestyle='--', alpha=0.3, label='R²=0.8 threshold')
    
    # Plot 2: R² vs RMSE scatter
    axes[1].scatter(results_df['Expert_RMSE'], results_df['Expert_R2'], 
                    label='Best Expert', marker='o', s=80, alpha=0.7, color='steelblue')
    axes[1].scatter(results_df['MoE_RMSE'], results_df['MoE_R2'],
                    label='MoE (Corrected)', marker='^', s=80, alpha=0.7, color='coral')
    
    for i, param in enumerate(params):
        axes[1].annotate(param, (results_df['Expert_RMSE'].iloc[i], results_df['Expert_R2'].iloc[i]),
                        fontsize=6, alpha=0.7)
    
    axes[1].set_xlabel('RMSE')
    axes[1].set_ylabel('R²')
    axes[1].set_title('R² vs. RMSE (Test Set)')
    axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'expert_vs_moe_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nExperiment completed at: {datetime.now()}")
print("All results and models saved.")